# LeetCode 1426 — Counting Elements bằng Python

> **Bài Premium: đề bài trong notebook được viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.**

**Link đề:** https://leetcode.com/problems/counting-elements/
**Độ khó:** Easy · **Chủ đề:** Array, Hash Table

```python
class Solution:
    def countElements(self, arr: list[int]) -> int:
        ...
```

## Mục tiêu học
1. Hiểu đề "đếm các phần tử `x` mà `x + 1` cũng có trong mảng" — **đếm cả bản trùng**.
2. Học `set`: tạo, kiểm tra `in` trung bình `O(1)`, khác `list` ra sao.
3. Đi từ vét cạn `O(n²)` → `set` `O(n)` → `Counter` (đếm theo giá trị) → sắp xếp + nhóm → mảng đếm.
4. Trace, chứng minh, Time/Space, lỗi hay gặp (đếm trùng sai, lặp trên `set` làm mất bản trùng).
5. Bài tập biến thể (`x − 1`, `x + k`, đếm cặp).

**Cách học:** đọc markdown → đoán output → chạy cell → so sánh. Cell TODO chỉ định nghĩa hàm, không tự chạy.

## 1. Đề bài

Cho mảng số nguyên `arr`. Đếm xem có **bao nhiêu phần tử** `x` sao cho `x + 1` **cũng nằm trong** `arr`.

Nếu `arr` có phần tử **trùng lặp**, mỗi bản được **đếm riêng**.

| `arr` | Output | Giải thích |
|---|---:|---|
| `[1,2,3]` | `2` | `1` (có `2`) và `2` (có `3`) được đếm; `3` thì không có `4` |
| `[1,1,3,3,5,5,7,7]` | `0` | không có `2, 4, 6, 8` |
| `[1,3,2,3,5,0]` | `3` | `0` (có `1`), `1` (có `2`), `2` (có `3`) |
| `[1,1,2,2]` | `2` | hai số `1` đều có `2`; hai số `2` không có `3` |

**Constraints (theo hiểu biết):** `1 <= arr.length <= 1000`, `0 <= arr[i] <= 1000`.

Chú ý ví dụ cuối: `x + 1` chỉ cần **tồn tại** (một lần là đủ), còn `x` thì đếm **mọi bản sao**.

In [1]:
examples = [
    ([1, 2, 3], 2),
    ([1, 1, 3, 3, 5, 5, 7, 7], 0),
    ([1, 3, 2, 3, 5, 0], 3),
    ([1, 1, 2, 2], 2),
]
for arr, expected in examples:
    print(f"arr = {arr} -> kỳ vọng {expected}")

arr = [1, 2, 3] -> kỳ vọng 2
arr = [1, 1, 3, 3, 5, 5, 7, 7] -> kỳ vọng 0
arr = [1, 3, 2, 3, 5, 0] -> kỳ vọng 3
arr = [1, 1, 2, 2] -> kỳ vọng 2


## 2. Nền tảng Python cần biết

### 2.1. `in` trên `list` — đúng nhưng chậm

`x in some_list` duyệt từ đầu tới khi gặp ⇒ trung bình/xấu nhất `O(n)`.

In [2]:
arr = [1, 3, 2, 3, 5, 0]
print(4 in arr, 2 in arr)

False True


### 2.2. `set` — tập hợp, tra cứu nhanh

`set(arr)` tạo **tập hợp**: mỗi giá trị chỉ giữ **một lần**, không có thứ tự cố định. `x in s` trung bình `O(1)` nhờ **bảng băm** (hash table).

In [3]:
s = set([1, 1, 2, 2, 5])
print("set:", s)
print("3 in s:", 3 in s, "| 2 in s:", 2 in s)
print("số phần tử khác nhau:", len(s))
s.add(9)
print("sau add(9):", s)

set: {1, 2, 5}
3 in s: False | 2 in s: True
số phần tử khác nhau: 3
sau add(9): {1, 2, 5, 9}


### 2.3. So sánh tốc độ `in` trên `list` và `set`

In [4]:
import time
big_list = list(range(100000))
big_set = set(big_list)
t0 = time.perf_counter(); [99999 in big_list for _ in range(200)]; t1 = time.perf_counter()
[99999 in big_set for _ in range(200)]; t2 = time.perf_counter()
print(f"list: {1000 * (t1 - t0):.2f} ms | set: {1000 * (t2 - t1):.4f} ms (200 lần tra cứu)")

list: 111.64 ms | set: 0.0985 ms (200 lần tra cứu)


### 2.4. `sum(1 for ... if ...)` — đếm số phần tử thỏa điều kiện

Generator sinh `1` cho mỗi phần tử thỏa ⇒ `sum` ra số lượng. Cách khác: `sum(điều_kiện for ...)` vì `True` = 1.

In [5]:
nums = [4, 7, 10, 3]
print(sum(1 for x in nums if x > 5))
print(sum(x > 5 for x in nums))

2
2


### 2.5. `collections.Counter` — đếm theo giá trị

`Counter(arr)` → dict `giá_trị → số lần`. Hữu ích khi muốn duyệt **giá trị khác nhau** nhưng vẫn biết số bản sao.

In [6]:
from collections import Counter
cnt = Counter([1, 1, 2, 2, 5])
print(cnt, "| cnt[1] =", cnt[1], "| cnt[4] =", cnt[4])

Counter({1: 2, 2: 2, 5: 1}) | cnt[1] = 2 | cnt[4] = 0


## 3. Cách trực giác (baseline) — với mỗi `x`, tìm `x + 1` trong cả mảng

```text
count = 0
với mỗi x trong arr:
    nếu (x + 1) có trong arr (duyệt list): count += 1
```

In [7]:
def count_elements_bruteforce(arr: list[int]) -> int:
    count = 0
    for x in arr:
        found = False
        for y in arr:
            if y == x + 1:
                found = True
                break
        if found:
            count += 1
    return count

for arr, expected in examples:
    print(arr, "->", count_elements_bruteforce(arr), "| kỳ vọng", expected)

[1, 2, 3] -> 2 | kỳ vọng 2
[1, 1, 3, 3, 5, 5, 7, 7] -> 0 | kỳ vọng 0
[1, 3, 2, 3, 5, 0] -> 3 | kỳ vọng 3
[1, 1, 2, 2] -> 2 | kỳ vọng 2


### 3.1. Nút thắt

Vòng trong duyệt lại toàn bộ mảng cho mỗi `x` ⇒ `O(n²)`. Với `n = 1000` là `10^6` phép so sánh — chạy được, nhưng câu hỏi "có `x + 1` không?" được hỏi đi hỏi lại trên **cùng một mảng không đổi**. Đây là tín hiệu: **tiền xử lý thành `set`** để mỗi câu hỏi chỉ `O(1)`.

In [8]:
def count_comparisons(arr):
    comps = 0
    for x in arr:
        for y in arr:
            comps += 1
            if y == x + 1:
                break
    return comps

worst = list(range(0, 2000, 2))       # 1000 số chẵn: không số nào có x+1 -> duyệt hết
print("n =", len(worst), "-> số phép so sánh:", count_comparisons(worst))

n = 1000 -> số phép so sánh: 1000000


## 4. Insight — đổi mảng thành `set` một lần

1. `present = set(arr)` — `O(n)`.
2. Duyệt **mảng gốc** (không phải `set`, để đếm đủ bản trùng): mỗi `x` kiểm tra `x + 1 in present` — `O(1)` trung bình.

Điểm then chốt: tra cứu trên `set`, nhưng **đếm** trên `list`.

In [9]:
arr = [1, 1, 2, 2]
present = set(arr)
print("present =", present)
for x in arr:
    print(f"x={x}: x+1={x + 1} in present? {x + 1 in present}")

present = {1, 2}
x=1: x+1=2 in present? True
x=1: x+1=2 in present? True
x=2: x+1=3 in present? False
x=2: x+1=3 in present? False


## 5. Lời giải chính — `set`, giảng từng đoạn

**Đoạn 1:** `present = set(arr)`.

**Đoạn 2:** `count = 0`; duyệt `x` trong `arr`.

**Đoạn 3:** nếu `x + 1 in present` thì `count += 1`.

**Đoạn 4:** trả `count`.

In [10]:
def count_elements_set(arr: list[int]) -> int:
    present = set(arr)            # Đoạn 1
    count = 0                     # Đoạn 2
    for x in arr:
        if x + 1 in present:      # Đoạn 3
            count += 1
    return count                  # Đoạn 4

for arr, expected in examples:
    print(arr, "->", count_elements_set(arr), "| kỳ vọng", expected)

[1, 2, 3] -> 2 | kỳ vọng 2
[1, 1, 3, 3, 5, 5, 7, 7] -> 0 | kỳ vọng 0
[1, 3, 2, 3, 5, 0] -> 3 | kỳ vọng 3
[1, 1, 2, 2] -> 2 | kỳ vọng 2


### 5.1. Bản một dòng

In [11]:
def count_elements_oneliner(arr: list[int]) -> int:
    present = set(arr)
    return sum(x + 1 in present for x in arr)

print([count_elements_oneliner(a) for a, _ in examples])

[2, 0, 3, 2]


## 6. Cách 3 — `Counter`: duyệt theo **giá trị khác nhau**

Nếu giá trị `v` xuất hiện `cnt[v]` lần và `v + 1` tồn tại, thì đóng góp `cnt[v]` vào kết quả. Số vòng lặp = số giá trị khác nhau `k` (có thể nhỏ hơn nhiều so với `n`).

In [12]:
def count_elements_counter(arr: list[int]) -> int:
    cnt = Counter(arr)
    return sum(c for v, c in cnt.items() if v + 1 in cnt)

for arr, expected in examples:
    print(arr, "->", count_elements_counter(arr), "| kỳ vọng", expected)

[1, 2, 3] -> 2 | kỳ vọng 2
[1, 1, 3, 3, 5, 5, 7, 7] -> 0 | kỳ vọng 0
[1, 3, 2, 3, 5, 0] -> 3 | kỳ vọng 3
[1, 1, 2, 2] -> 2 | kỳ vọng 2


`v + 1 in cnt` kiểm tra **key** có trong `Counter` (giống dict). Chú ý: `cnt[v + 1]` (truy cập) sẽ trả `0` nếu thiếu — nhưng **không** thêm key; còn `in` mới là cách kiểm tra tồn tại rõ ràng.

## 7. Cách 4 — sắp xếp rồi đếm theo nhóm (không dùng hash)

Sắp tăng dần: các giá trị bằng nhau liền nhau. Duyệt từng **nhóm** `[i, j)` cùng giá trị `v`; nếu nhóm kế tiếp có giá trị `v + 1` thì cộng độ dài nhóm `j − i`.

In [13]:
def count_elements_sort(arr: list[int]) -> int:
    a = sorted(arr)
    n = len(a)
    count = 0
    i = 0
    while i < n:
        j = i
        while j < n and a[j] == a[i]:      # nhóm [i, j) cùng giá trị a[i]
            j += 1
        if j < n and a[j] == a[i] + 1:     # nhóm kế là v + 1
            count += j - i
        i = j
    return count

for arr, expected in examples:
    print(arr, "->", count_elements_sort(arr), "| kỳ vọng", expected)

[1, 2, 3] -> 2 | kỳ vọng 2
[1, 1, 3, 3, 5, 5, 7, 7] -> 0 | kỳ vọng 0
[1, 3, 2, 3, 5, 0] -> 3 | kỳ vọng 3
[1, 1, 2, 2] -> 2 | kỳ vọng 2


### 7.1. Cách 5 — mảng đếm theo constraint `0 <= arr[i] <= 1000`

`freq[v]` = số lần `v` xuất hiện; đáp án = tổng `freq[v]` với `freq[v + 1] > 0`. Cần mảng dài `1002` để `v + 1` không vượt chỉ số khi `v = 1000`.

In [14]:
def count_elements_array(arr: list[int]) -> int:
    freq = [0] * 1002
    for x in arr:
        freq[x] += 1
    return sum(freq[v] for v in range(1001) if freq[v + 1] > 0)

for arr, expected in examples:
    print(arr, "->", count_elements_array(arr), "| kỳ vọng", expected)

[1, 2, 3] -> 2 | kỳ vọng 2
[1, 1, 3, 3, 5, 5, 7, 7] -> 0 | kỳ vọng 0
[1, 3, 2, 3, 5, 0] -> 3 | kỳ vọng 3
[1, 1, 2, 2] -> 2 | kỳ vọng 2


## 8. TRACE từng bước

Trace cách `set` với `arr = [1, 3, 2, 3, 5, 0]`, `present = {0, 1, 2, 3, 5}`:

| i | x | x + 1 | x + 1 in present? | count |
|---:|---:|---:|:---:|---:|
| 0 | 1 | 2 | ✔ | 1 |
| 1 | 3 | 4 | ✘ | 1 |
| 2 | 2 | 3 | ✔ | 2 |
| 3 | 3 | 4 | ✘ | 2 |
| 4 | 5 | 6 | ✘ | 2 |
| 5 | 0 | 1 | ✔ | 3 |

Kết quả `3`.

In [15]:
def trace_count_elements(arr: list[int]) -> int:
    present = set(arr)
    print(f"arr = {arr}, present = {sorted(present)}")
    count = 0
    for i, x in enumerate(arr):
        hit = x + 1 in present
        if hit:
            count += 1
        print(f"  i={i}, x={x}, x+1={x + 1}, có? {hit!s:<5} -> count={count}")
    return count

print("=>", trace_count_elements([1, 3, 2, 3, 5, 0]))
print()
print("=>", trace_count_elements([1, 1, 2, 2]))

arr = [1, 3, 2, 3, 5, 0], present = [0, 1, 2, 3, 5]
  i=0, x=1, x+1=2, có? True  -> count=1
  i=1, x=3, x+1=4, có? False -> count=1
  i=2, x=2, x+1=3, có? True  -> count=2
  i=3, x=3, x+1=4, có? False -> count=2
  i=4, x=5, x+1=6, có? False -> count=2
  i=5, x=0, x+1=1, có? True  -> count=3
=> 3

arr = [1, 1, 2, 2], present = [1, 2]
  i=0, x=1, x+1=2, có? True  -> count=1
  i=1, x=1, x+1=2, có? True  -> count=2
  i=2, x=2, x+1=3, có? False -> count=2
  i=3, x=2, x+1=3, có? False -> count=2
=> 2


Ví dụ `[1, 1, 2, 2]` cho thấy nhánh **trùng lặp**: cả hai số `1` đều được cộng.

Trace cách sắp xếp + nhóm với cùng ví dụ trùng:

In [16]:
def trace_count_sort(arr):
    a = sorted(arr)
    print("sorted:", a)
    n, i, count = len(a), 0, 0
    while i < n:
        j = i
        while j < n and a[j] == a[i]:
            j += 1
        nxt = a[j] if j < n else None
        add = j - i if (j < n and a[j] == a[i] + 1) else 0
        count += add
        print(f"  nhóm giá trị {a[i]} (dài {j - i}), nhóm kế = {nxt} -> cộng {add}, count = {count}")
        i = j
    return count

print("=>", trace_count_sort([1, 1, 2, 2, 4, 5, 5]))

sorted: [1, 1, 2, 2, 4, 5, 5]
  nhóm giá trị 1 (dài 2), nhóm kế = 2 -> cộng 2, count = 2
  nhóm giá trị 2 (dài 2), nhóm kế = 4 -> cộng 0, count = 2
  nhóm giá trị 4 (dài 1), nhóm kế = 5 -> cộng 1, count = 3
  nhóm giá trị 5 (dài 2), nhóm kế = None -> cộng 0, count = 3
=> 3


## 9. Code nộp LeetCode

In [17]:
class Solution:
    def countElements(self, arr: list[int]) -> int:
        present = set(arr)
        count = 0
        for x in arr:
            if x + 1 in present:
                count += 1
        return count

print([Solution().countElements(a) for a, _ in examples])

[2, 0, 3, 2]


### Bản `Counter`

In [18]:
class SolutionCounter:
    def countElements(self, arr: list[int]) -> int:
        cnt = Counter(arr)
        return sum(c for v, c in cnt.items() if v + 1 in cnt)

print([SolutionCounter().countElements(a) for a, _ in examples])

[2, 0, 3, 2]


Trên LeetCode, chữ ký gốc thường là `def countElements(self, arr: List[int]) -> int:` với `from typing import List` — tương đương `list[int]`.

### Bảng so sánh

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| 1. Vét cạn | tìm `x+1` bằng vòng trong | `O(n²)` | `O(1)` | dễ nghĩ |
| 2. `set` | tra cứu `O(1)` trung bình | `O(n)` | `O(n)` | chuẩn |
| 3. `Counter` | duyệt giá trị khác nhau | `O(n)` | `O(k)` | `k` = số giá trị khác nhau |
| 4. Sắp xếp + nhóm | nhóm kế là `v+1` | `O(n log n)` | `O(n)` | không cần hash |
| 5. Mảng đếm | `freq[v]`, `freq[v+1]` | `O(n + V)` | `O(V)` | `V = 1001` |

## 10. Vì sao đúng?

**Cách `set`:** `present` chứa đúng các giá trị có trong `arr`. Với mỗi **vị trí** `i`, ta cộng 1 khi và chỉ khi `arr[i] + 1 ∈ present` ⇔ `arr[i] + 1` có trong `arr`. Duyệt mọi vị trí ⇒ mỗi bản sao được xét riêng ⇒ đúng định nghĩa "đếm riêng các bản trùng".

**Cách `Counter`:** gom các vị trí có cùng giá trị `v` thành một nhóm `cnt[v]`; điều kiện chỉ phụ thuộc `v`, nên cả nhóm cùng được đếm hoặc cùng bị loại ⇒ tổng `cnt[v]` trên các `v` thỏa = kết quả cách `set`.

**Cách sắp xếp:** sau khi sắp, nếu `v + 1` tồn tại thì nhóm của nó đứng **ngay sau** nhóm `v` (không có số nguyên nào giữa `v` và `v + 1`). Vì vậy chỉ cần nhìn nhóm kế tiếp.

## 11. Time / Space

Gọi `n = len(arr)`, `k` = số giá trị khác nhau, `V = 1001`.

- `set`: dựng `O(n)` + `n` lần tra `O(1)` trung bình ⇒ **`O(n)`**; bộ nhớ `O(k)`.
- `Counter`: `O(n)` dựng + `O(k)` duyệt.
- Sắp xếp: `O(n log n)`; bộ nhớ `O(n)` cho bản sao.
- Mảng đếm: `O(n + V)` thời gian, `O(V)` bộ nhớ.
- Vét cạn: `O(n²)`.

In [19]:
import random
random.seed(1)
big = [random.randint(0, 1000) for _ in range(1000)]
for f in [count_elements_bruteforce, count_elements_set, count_elements_counter, count_elements_sort, count_elements_array]:
    t0 = time.perf_counter()
    for _ in range(20):
        r = f(big)
    print(f"{f.__name__:<26} -> {r} ({1000 * (time.perf_counter() - t0):7.2f} ms / 20 lần)")

count_elements_bruteforce  -> 635 ( 370.57 ms / 20 lần)
count_elements_set         -> 635 (   1.44 ms / 20 lần)
count_elements_counter     -> 635 (   1.72 ms / 20 lần)
count_elements_sort        -> 635 (   4.47 ms / 20 lần)
count_elements_array       -> 635 (   1.76 ms / 20 lần)


## 12. Bộ test

Mẫu; một phần tử; chứa `0`; giá trị biên `1000`; mọi phần tử bằng nhau; dãy liên tiếp `0..999`; trùng lặp nhiều; và **đối chiếu ngẫu nhiên** mọi cách với vét cạn.

In [20]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<24} PASS -> {actual}")

sol = Solution()
check("ví dụ 1", sol.countElements([1, 2, 3]), 2)
check("ví dụ 2", sol.countElements([1, 1, 3, 3, 5, 5, 7, 7]), 0)
check("ví dụ 3", sol.countElements([1, 3, 2, 3, 5, 0]), 3)
check("ví dụ 4 (trùng)", sol.countElements([1, 1, 2, 2]), 2)
check("một phần tử", sol.countElements([5]), 0)
check("chứa 0", sol.countElements([0, 1]), 1)
check("biên 1000", sol.countElements([999, 1000, 1000]), 1)
check("toàn bằng nhau", sol.countElements([4] * 10), 0)
check("dãy 0..999", sol.countElements(list(range(1000))), 999)
check("trùng nhiều bên trái", sol.countElements([2, 2, 2, 3]), 3)

ví dụ 1                  PASS -> 2
ví dụ 2                  PASS -> 0
ví dụ 3                  PASS -> 3
ví dụ 4 (trùng)          PASS -> 2
một phần tử              PASS -> 0
chứa 0                   PASS -> 1
biên 1000                PASS -> 1
toàn bằng nhau           PASS -> 0
dãy 0..999               PASS -> 999
trùng nhiều bên trái     PASS -> 3


In [21]:
methods = [count_elements_set, count_elements_oneliner, count_elements_counter, count_elements_sort,
           count_elements_array, Solution().countElements, SolutionCounter().countElements]
random.seed(1426)
for _ in range(3000):
    arr = [random.randint(0, 12) for _ in range(random.randint(1, 15))]
    expected = count_elements_bruteforce(arr)
    for f in methods:
        assert f(arr) == expected, (f, arr)
print("Mọi cách khớp vét cạn trên 3000 mảng ngẫu nhiên")

Mọi cách khớp vét cạn trên 3000 mảng ngẫu nhiên


## 13. Lỗi người mới hay gặp

### Lỗi 1: Duyệt trên `set` thay vì `list` → mất bản trùng

In [22]:
def wrong_iterate_set(arr):
    present = set(arr)
    return sum(1 for x in present if x + 1 in present)   # SAI: mỗi giá trị chỉ đếm 1 lần

print("sai:", wrong_iterate_set([1, 1, 2, 2]), "| đúng:", count_elements_set([1, 1, 2, 2]))

sai: 1 | đúng: 2


### Lỗi 2: Đếm theo số bản sao của `x + 1` thay vì của `x`

In [23]:
def wrong_count_successor(arr):
    cnt = Counter(arr)
    return sum(cnt[v + 1] for v in cnt if v + 1 in cnt)   # SAI: dùng cnt[v+1]

print("sai:", wrong_count_successor([2, 3, 3, 3]), "| đúng:", count_elements_set([2, 3, 3, 3]))

sai: 3 | đúng: 1


### Lỗi 3: Tra `x + 1` trong `list` bên trong vòng lặp

`if x + 1 in arr` — đúng nhưng `O(n²)`. Với `n` lớn hơn (ví dụ `10^5`) sẽ **TLE** (Time Limit Exceeded). Luôn đổi sang `set` trước.

### Lỗi 4: Mảng đếm thiếu ô cho `v + 1`

`freq = [0] * 1001` rồi truy cập `freq[1001]` khi `v = 1000` → `IndexError`.

In [24]:
try:
    freq = [0] * 1001
    v = 1000
    print(freq[v + 1])
except IndexError as e:
    print("IndexError:", e, "-> cần [0] * 1002 hoặc giới hạn range")

IndexError: list index out of range -> cần [0] * 1002 hoặc giới hạn range


## 14. Mẹo nhận dạng pattern

- "Có tồn tại phần tử `f(x)` trong mảng không?" (`x + 1`, `2x`, `target − x`) ⇒ **dựng `set` một lần**, tra `O(1)`.
- "Đếm riêng bản trùng" ⇒ duyệt **mảng gốc** (hoặc dùng `Counter` và nhân theo số lần).
- Miền giá trị nhỏ ⇒ có thể thay `set` bằng **mảng đếm/mảng boolean**.
- Quan hệ "liền kề giá trị" (`v`, `v+1`) ⇒ sau khi **sắp xếp**, chúng nằm cạnh nhau.

## 15. Biến thể

1. Đếm `x` mà `x − 1` có trong mảng.
2. Tổng quát: `x + k` có trong mảng.
3. Đếm **cặp chỉ số** `(i, j)` với `arr[j] = arr[i] + 1`.
4. LeetCode 1346 *Check If N and Its Double Exist* (`2x` có trong mảng), LeetCode 128 *Longest Consecutive Sequence* (dùng `set` + "chỉ bắt đầu từ `x` khi `x − 1` không có").

In [25]:
def count_plus_k(arr, k):
    present = set(arr)
    return sum(x + k in present for x in arr)

def count_pairs_plus_one(arr):
    cnt = Counter(arr)
    return sum(c * cnt[v + 1] for v, c in cnt.items())

def longest_consecutive(nums):                # LeetCode 128
    s = set(nums)
    best = 0
    for x in s:
        if x - 1 not in s:                    # x là đầu một dãy liên tiếp
            y = x
            while y + 1 in s:
                y += 1
            best = max(best, y - x + 1)
    return best

print(count_plus_k([1, 3, 2, 3, 5, 0], -1), count_plus_k([1, 3, 5, 7], 2))
print(count_pairs_plus_one([1, 1, 2, 2]))
print(longest_consecutive([100, 4, 200, 1, 3, 2]))

4 3
4
4


## 16. Bài tập tự luyện (TODO — tăng dần)

**Bài 1 (tính tay):** `arr = [3, 4, 4, 5, 7, 8, 8]`. Đáp án là bao nhiêu? Liệt kê những phần tử được đếm.

**Bài 2 (đoán output):** `set([2, 2, 3])`, `len(set("banana"))`, `4 in {1, 4}`, `sum(x > 1 for x in [0, 2, 5])`.

**Bài 3 (điền chỗ trống):** `exercise_count_elements(arr)` dùng `set`.

**Bài 4 (tự viết):** `exercise_count_minus_one(arr)` — đếm `x` mà `x − 1` có trong mảng.

**Bài 5 (biến thể):** `exercise_count_pairs(arr)` — số cặp chỉ số `(i, j)` (`i != j`, có thể `i > j`) sao cho `arr[j] == arr[i] + 1`.

**Bài 6 (mini challenge):** `exercise_longest_chain(arr)` — độ dài dãy giá trị liên tiếp dài nhất (`v, v+1, ..., v+L−1` đều có trong mảng), `O(n)` trung bình.

In [26]:
def exercise_count_elements(arr: list[int]) -> int:
    # TODO 1: present = set(arr)
    # TODO 2: duyệt arr (không phải present!)
    # TODO 3: đếm x + 1 in present
    pass

def exercise_count_minus_one(arr: list[int]) -> int:
    # TODO
    pass

# assert exercise_count_elements([1, 1, 2, 2]) == 2
# assert exercise_count_minus_one([1, 2, 2, 3]) == 3

In [27]:
def exercise_count_pairs(arr: list[int]) -> int:
    # TODO
    pass

def exercise_longest_chain(arr: list[int]) -> int:
    # TODO
    pass

# assert exercise_count_pairs([1, 1, 2, 2]) == 4
# assert exercise_longest_chain([100, 4, 200, 1, 3, 2]) == 4

## 17. Gợi ý 3 mức

**Bài 1** — (1) `present = {3, 4, 5, 7, 8}`. (2) Xét từng phần tử: `3→4` có, `4→5` có (2 lần), `5→6` không, `7→8` có, `8→9` không. (3) Cộng lại.

**Bài 3** — (1) Dựng `set`. (2) Duyệt `arr`. (3) `if x + 1 in present: count += 1`.

**Bài 4** — (1) Giống bài 3. (2) Đổi `x + 1` thành `x − 1`. (3) Chú ý `x = 0` → `−1` không có, không lỗi gì cả.

**Bài 5** — (1) Mỗi bản sao của `v` ghép được với mọi bản sao của `v + 1`. (2) `Counter`. (3) `sum(cnt[v] * cnt[v + 1])`.

**Bài 6** — (1) `set`. (2) Chỉ bắt đầu đếm từ `x` khi `x − 1` **không** có (đầu dãy). (3) Đi `x, x+1, ...` tới khi hết; mỗi phần tử được đi qua tối đa một lần ⇒ `O(n)`.

## 18. Đáp án tham khảo

**Bài 1:** được đếm: `3`, `4`, `4`, `7` ⇒ **4**.

**Bài 2:** `{2, 3}`, `3`, `True`, `2`.

In [28]:
def answer_count_elements(arr):
    present = set(arr)
    count = 0
    for x in arr:
        if x + 1 in present:
            count += 1
    return count

def answer_count_minus_one(arr):
    present = set(arr)
    return sum(x - 1 in present for x in arr)

def answer_count_pairs(arr):
    cnt = Counter(arr)
    return sum(c * cnt[v + 1] for v, c in cnt.items())

def answer_longest_chain(arr):
    s = set(arr)
    best = 0
    for x in s:
        if x - 1 not in s:
            y = x
            while y + 1 in s:
                y += 1
            best = max(best, y - x + 1)
    return best

def brute_pairs(arr):
    return sum(1 for i in range(len(arr)) for j in range(len(arr)) if i != j and arr[j] == arr[i] + 1)

assert answer_count_elements([3, 4, 4, 5, 7, 8, 8]) == 4
assert set([2, 2, 3]) == {2, 3} and len(set("banana")) == 3 and (4 in {1, 4}) and sum(x > 1 for x in [0, 2, 5]) == 2
assert answer_count_elements([1, 1, 2, 2]) == 2
assert answer_count_minus_one([1, 2, 2, 3]) == 3
assert answer_count_pairs([1, 1, 2, 2]) == 4
assert answer_longest_chain([100, 4, 200, 1, 3, 2]) == 4
random.seed(8)
for _ in range(1000):
    arr = [random.randint(0, 10) for _ in range(random.randint(1, 12))]
    assert answer_count_elements(arr) == count_elements_bruteforce(arr)
    assert answer_count_pairs(arr) == brute_pairs(arr)
    vals = set(arr)
    brute_chain = max(L for v in vals for L in range(1, 12) if all(v + t in vals for t in range(L)))
    assert answer_longest_chain(arr) == brute_chain
print("Tất cả đáp án PASS!")

Tất cả đáp án PASS!


## 19. Template nhớ nhanh + checklist submit

```python
class Solution:
    def countElements(self, arr: list[int]) -> int:
        present = set(arr)                        # tra cứu O(1)
        return sum(x + 1 in present for x in arr) # đếm trên mảng gốc (giữ bản trùng)
```

**Khung tư duy "tồn tại phần tử liên quan":** dựng `set` → với mỗi phần tử tra `f(x) in set` → đếm/trả về.

**Checklist:**
- [ ] Đếm trên **mảng gốc** để giữ bản trùng.
- [ ] Tra cứu trên **`set`**, không phải `list`.
- [ ] Đếm theo số bản sao của **`x`**, không phải của `x + 1`.
- [ ] Mảng đếm (nếu dùng) đủ chỗ cho `v + 1`.
- [ ] (Premium) Đối chiếu lại đề/constraints nếu có tài khoản.